In [ ]:
import os
from time import time
import numpy as np
import matplotlib.pyplot as plt

from theia import elevationAt, Point, Radar, radar_eq_max_dist, calculate_coverage
from theia.detection.active import calculate_snr
from theia.line_of_sight import has_line_of_sight
from theia.doppler import monostatic_doppler
from theia.types import Polarization
from theia.data_loading import load_trajectory_file

# Evaluating digital elevation model (DEM)

In [ ]:
# Call once to precompile and cache.
elevationAt(47.671, 8.5325)

In [ ]:
%timeit elevationAt(47.671, 8.5325)

In [ ]:
import math

from theia.config import ELEVATION_DATA_DIR
from theia.data_loading import load_hgt_file
from theia.data_loading import interpolate_elevation_tile

In [ ]:
%%timeit
lat = 47.671
lon = 8.5325

lat0 = math.floor(lat)
lon0 = math.floor(lon)

ns = "N" if lat0 >= 0 else "S"
ew = "E" if lon0 >= 0 else "W"
filename = f"{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt"

arr = load_hgt_file(f"{ELEVATION_DATA_DIR}/{filename}")

# local fractional degree within tile
lat_f = lat - lat0
lon_f = lon - lon0

In [ ]:
%%timeit
lat = 47.671
lon = 8.5325

lat0 = math.floor(lat)
lon0 = math.floor(lon)

ns = "N" if lat0 >= 0 else "S"
ew = "E" if lon0 >= 0 else "W"

filename = f"{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt"

lat_f = lat - lat0
lon_f = lon - lon0

In [ ]:
# %%timeit
# load_hgt_file(f"{ELEVATION_DATA_DIR}/{filename}")

In [ ]:
# %%timeit
# interpolate_elevation_tile(lat_f, lon_f, arr)

# Radar equation

In [ ]:
from theia.types import Receiver, Transmitter


p = Point(
    lat=47.36700085728634,
    lon=8.537724304199216,
    alt=408,
)
antenna_height = 10.0
radar = Radar(
    transmitter=Transmitter(
        id=585,
        point=p,
        power=20000,
        erp=800,
        antenna_height=antenna_height,
        antenna_diameter=2.0,
        frequency=1000.0,
        pulse_width=1,
        polarization=Polarization.HORIZONTAL,
        bandwidth=1,
    ),
    receiver=Receiver(
        id=585,
        point=p,
        antenna_height=antenna_height,
        diameter=2.0,
        cpi_pulses=1,
        pfa=1e-6,
        min_elevation=-20.0,
        max_elevation=60.0,
        rotation_time=10.0,
        bandwidth=1,
    ),
)

target_rcs = 2.0

In [ ]:
max_dist = radar_eq_max_dist(radar, target_rcs)

In [ ]:
%timeit radar_eq_max_dist(radar, target_rcs)

# Coverage

In [ ]:
start = Point(
    lat=radar.transmitter.lat,
    lon=radar.transmitter.lon,
    alt=elevationAt(
        radar.transmitter.lat,
        radar.transmitter.lon,
    ),
)

theta = 30.0
target_alt = 1000.0

In [ ]:
resolutions = [50, 100, 200, 400, 800]

times = []
results = []
for res in resolutions:
    t_start = time()
    coverage = calculate_coverage(
        start,
        max_dist,
        1000.0,
        d_theta=0.1,
        dist_res=res,
    )
    t_stop = time()
    times.append(t_stop - t_start)
    results.append(coverage)

In [ ]:
import json
import folium

map = folium.Map()

folium.GeoJson(results[0]).add_to(map)

with open("../tests/test_data/reference_coverage.geojson", "r") as file:
    ref = json.load(file)

folium.GeoJson(ref, color="red").add_to(map)

map.save("map.html")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(resolutions, times, "o-", label="Measured")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=16)
ax.set_xlabel("Resolution [m]", fontsize=20)
ax.set_ylabel("Runtime [s]", fontsize=20)
# ax.set_xscale("log")
# ax.set_yscale("log")

# Exponential fit.
params = np.polyfit(np.log(resolutions), np.log(times), deg=1)
f = lambda res: np.exp(np.polyval(params, np.log(resolutions)))

ax.plot(resolutions, f(resolutions), "o-", label=f"Fit runtime ~ res^{params[0]:.2f}")

fig.legend()
fig.tight_layout()

In [ ]:
diff = results[-1].difference(results[0])
diff

In [ ]:
print(
    f"Relative area difference res. {resolutions[0]}m vs. {resolutions[-1]}m: {(diff.area / results[0].area) * 100.0:.2f}%"
)

# Active radar detection

In [ ]:
calculated = calculate_snr(
    1.0,
    2.0,
    7.064422898951063,
    20_000.0,
    1.0,
    1.0,
    2.0,
    1,
    300.0,
    1.9,
    12.0,
)

In [ ]:
%%timeit
calculated = calculate_snr(
    1.0,
    2.0,
    7.064422898951063,
    20_000.0,
    1.0,
    1.0,
    2.0,
    1,
    300.0,
    1.9,
    12.0,
)

In [ ]:
from theia.detection.active import get_rad_pd
from theia.types import Radar, Target, Polarization, Point, Velocity

transmitter = radar.transmitter
target = Target(
    id=1,
    point=Point(lat=47.348, lon=8.6266, alt=1000.0),
    cross_section=1.0,
    velocity=Velocity(vx=250.0, vy=0.0, vz=0.0),
)

# Call once to precomile etc.
get_rad_pd(radar, target, 30.0, 5.0, 12.0)

In [ ]:
%%timeit
get_rad_pd(radar, target, 30.0, 5.0, 12.0)

In [ ]:
%%timeit
has_line_of_sight(transmitter.point, target.point, 30.0)

# Monostatic Doppler

In [ ]:
monostatic_doppler(
    1000.0,
    47.36700085728634,
    8.537724304199216,
    407.83600886023686,
    47.348,
    8.6266,
    1000.0,
    250.0,
    0.0,
    0.0,
)

In [ ]:
%%timeit
calculated = monostatic_doppler(
    1000.0,
    47.36700085728634,
    8.537724304199216,
    407.83600886023686,
    47.348,
    8.6266,
    1000.0,
    250.0,
    0.0,
    0.0,
)